<a href="https://colab.research.google.com/github/naksidil/rag-pdf-comparison-nutrition-poetry/blob/main/RAG_Nutrition_vs_Poetry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
# Perform Google Colab installs (if running in Google Colab)
import os

if "COLAB_GPU" in os.environ:
    print("[INFO] Running in Google Colab, installing requirements.")
    !pip install PyMuPDF # for reading PDFs with Python
    !pip install tqdm # for progress bars
    !pip install sentence-transformers # for embedding models
    !pip install accelerate # for quantization model loading
    !pip install bitsandbytes # for quantizing models (less storage space)


[INFO] Running in Google Colab, installing requirements.


In [26]:
# Download PDF file
import os
import requests

# Get PDF document
pdf_path = "human-nutrition-text.pdf"

# Download PDF if it doesn't already exist
if not os.path.exists(pdf_path):
  print("File doesn't exist, downloading...")

  # The URL of the PDF you want to download
  url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"

  # The local filename to save the downloaded file
  filename = pdf_path

  # Send a GET request to the URL
  response = requests.get(url)

  # Check if the request was successful
  if response.status_code == 200:
      # Open a file in binary write mode and save the content to it
      with open(filename, "wb") as file:
          file.write(response.content)
      print(f"The file has been downloaded and saved as {filename}")
  else:
      print(f"Failed to download the file. Status code: {response.status_code}")
else:
  print(f"File {pdf_path} exists.")

File human-nutrition-text.pdf exists.


In [27]:
# Requires !pip install PyMuPDF, see: https://github.com/pymupdf/pymupdf
import fitz # (pymupdf, found this is better than pypdf for our use case, note: licence is AGPL-3.0, keep that in mind if you want to use any code commercially)
from tqdm.auto import tqdm # for progress bars, requires !pip install tqdm

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text = text.replace("\n", " ").strip() # note: this might be different for each doc (best to experiment)

    # Other potential text formatting functions can go here
    return cleaned_text

# Open PDF and get lines/pages
# Note: this only focuses on text, rather than images/figures etc
def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """
    Opens a PDF file, reads its text content page by page, and collects statistics.

    Parameters:
        pdf_path (str): The file path to the PDF document to be opened and read.

    Returns:
        list[dict]: A list of dictionaries, each containing the page number
        (adjusted), character count, word count, sentence count, token count, and the extracted text
        for each page.
    """
    doc = fitz.open(pdf_path)  # open a document
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):  # iterate the document pages
        text = page.get_text()  # get plain text encoded as UTF-8
        text = text_formatter(text)
        pages_and_texts.append({"page_number": page_number - 41,  # adjust page numbers since our PDF starts on page 42
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count_raw": len(text.split(". ")),
                                "page_token_count": len(text) / 4,  # 1 token = ~4 chars, see: https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them
                                "text": text})
    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': -41,
  'page_char_count': 29,
  'page_word_count': 4,
  'page_sentence_count_raw': 1,
  'page_token_count': 7.25,
  'text': 'Human Nutrition: 2020 Edition'},
 {'page_number': -40,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''}]

In [28]:
pages_and_texts[50]

{'page_number': 9,
 'page_char_count': 1320,
 'page_word_count': 215,
 'page_sentence_count_raw': 4,
 'page_token_count': 330.0,
 'text': 'Minerals  Major Functions  Macro  Sodium  Fluid balance, nerve transmission, muscle contraction  Chloride  Fluid balance, stomach acid production  Potassium  Fluid balance, nerve transmission, muscle contraction  Calcium  Bone and teeth health maintenance, nerve transmission,  muscle contraction, blood clotting  Phosphorus  Bone and teeth health maintenance, acid-base balance  Magnesium  Protein production, nerve transmission, muscle  contraction  Sulfur  Protein production  Trace  Iron  Carries oxygen, assists in energy production  Zinc  Protein and DNA production, wound healing, growth,  immune system function  Iodine  Thyroid hormone production, growth, metabolism  Selenium  Antioxidant  Copper  Coenzyme, iron metabolism  Manganese  Coenzyme  Fluoride  Bone and teeth health maintenance, tooth decay  prevention  Chromium  Assists insulin in glucos

In [29]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,1,0.00,
2,-39,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,145,2,199.25,Contents Preface University of Hawai‘i at Mā...


In [30]:
import random

random.sample(pages_and_texts, k=3)

[{'page_number': 1026,
  'page_char_count': 1504,
  'page_word_count': 238,
  'page_sentence_count_raw': 17,
  'page_token_count': 376.0,
  'text': 'creating products that have a much longer shelf life than raw foods.  Also, food processing protects the health of the consumer and  allows for easier shipment and the marketing of foods by  corporations. However, there are certain drawbacks. Food  processing can reduce the nutritional content of raw ingredients.  For example, canning involves the use of heat, which destroys the  vitamin C in canned fruit. Also, certain food additives that are  included during processing, such as high fructose corn syrup, can  affect the health of a consumer. However, the level of added sugar  can make a major difference. Small amounts of added sugar and  other sweeteners, about 6 to 9 teaspoons a day or less, are not  considered harmful.1  Food Additives  If you examine the label for a processed food product, it is not  unusual to see a long list of added

In [31]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,1,0.00,
2,-39,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,145,2,199.25,Contents Preface University of Hawai‘i at Mā...


In [32]:
# Get stats
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00
std,348.86,560.38,95.76,6.19,140.10
min,-41.00,0.00,1.00,1.00,0.00
25%,260.75,762.00,134.00,4.00,190.50
50%,562.50,1231.50,214.50,10.00,307.88
75%,864.25,1603.50,271.00,14.00,400.88
max,1166.00,2308.00,429.00,32.00,577.00


In [33]:
from spacy.lang.en import English # see https://spacy.io/usage for install instructions

nlp = English()

# Add a sentencizer pipeline, see https://spacy.io/api/sentencizer/
nlp.add_pipe("sentencizer")

# Create a document instance as an example
doc = nlp("This is a sentence. This another sentence.")
assert len(list(doc.sents)) == 2

# Access the sentences of the document
list(doc.sents)

[This is a sentence., This another sentence.]

In [34]:
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)

    # Make sure all sentences are strings
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]

    # Count the sentences
    item["page_sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [35]:
# Inspect an example
import random
random.sample(pages_and_texts, k=1)

[{'page_number': 238,
  'page_char_count': 505,
  'page_word_count': 86,
  'page_sentence_count_raw': 2,
  'page_token_count': 126.25,
  'text': 'Image by  Allison  Calabrese /  CC BY 4.0  Learning Activities  Technology Note: The second edition of the Human  Nutrition Open Educational Resource (OER) textbook  features interactive learning activities.\xa0 These activities are  available in the web-based textbook and not available in the  downloadable versions (EPUB, Digital PDF, Print_PDF, or  Open Document).  Learning activities may be used across various mobile  devices, however, for the best user experience it is strongly  238  |  Introduction',
  'sentences': ['Image by  Allison  Calabrese /  CC BY 4.0  Learning Activities  Technology Note: The second edition of the Human  Nutrition Open Educational Resource (OER) textbook  features interactive learning activities.',
   '\xa0 These activities are  available in the web-based textbook and not available in the  downloadable versions (

In [36]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00,10.32
std,348.86,560.38,95.76,6.19,140.10,6.30
min,-41.00,0.00,1.00,1.00,0.00,0.00
25%,260.75,762.00,134.00,4.00,190.50,5.00
50%,562.50,1231.50,214.50,10.00,307.88,10.00
75%,864.25,1603.50,271.00,14.00,400.88,15.00
max,1166.00,2308.00,429.00,32.00,577.00,28.00


In [37]:
# Define split size to turn groups of sentences into chunks
num_sentence_chunk_size = 10

# Create a function that recursively splits a list into desired sizes
def split_list(input_list: list,
               slice_size: int) -> list[list[str]]:
    """
    Splits the input_list into sublists of size slice_size (or as close as possible).

    For example, a list of 17 sentences would be split into two lists of [[10], [7]]
    """
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

# Loop through pages and texts and split sentences into chunks
for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list=item["sentences"],
                                         slice_size=num_sentence_chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [38]:
# Sample an example from the group (note: many samples have only 1 chunk as they have <=10 sentences total)
random.sample(pages_and_texts, k=1)

[{'page_number': 1065,
  'page_char_count': 1427,
  'page_word_count': 249,
  'page_sentence_count_raw': 22,
  'page_token_count': 356.75,
  'text': 'reviews of randomized clinical trials reported that on average,  obesity treatments cause weight gain.56\xa0 This additional weight gain  leads to an increase in the set point, making it more difficult for an  individual to lose weight in the future. \xa0 Others reported a 3-5 %  weight loss was possible 4 years later if participants continued all  aspects of treatment.7\xa0\xa0For a 200 pound person, this represents a  6-10 pound weight loss.\xa0 The health benefits of this modest weight  loss are unclear and it is far less what is expected or desired when  following a diet.\xa0 In conclusion, the diet industry makes money from  a product that is proven not to work.  5.\xa0Mann, T., Tomiyama, A. J., Westling, E., Lew, A.-M.,  Samuels, B., & Chatman, J. (2007). Medicare’s search for  effective obesity treatments: Diets are not the answer.

In [39]:
# Create a DataFrame to get stats
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00,10.32,1.53
std,348.86,560.38,95.76,6.19,140.10,6.30,0.64
min,-41.00,0.00,1.00,1.00,0.00,0.00,0.00
25%,260.75,762.00,134.00,4.00,190.50,5.00,1.00
50%,562.50,1231.50,214.50,10.00,307.88,10.00,1.00
75%,864.25,1603.50,271.00,14.00,400.88,15.00,2.00
max,1166.00,2308.00,429.00,32.00,577.00,28.00,3.00


In [40]:
import re

# Split each chunk into its own item
pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]

        # Join the sentences together into a paragraph-like structure, aka a chunk (so they are a single string)
        joined_sentence_chunk = "".join(sentence_chunk).replace("  ", " ").strip()
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk) # ".A" -> ". A" for any full-stop/capital letter combo
        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        # Get stats about the chunk
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4 # 1 token = ~4 characters

        pages_and_chunks.append(chunk_dict)

# How many chunks do we have?
len(pages_and_chunks)

  0%|          | 0/1208 [00:00<?, ?it/s]

1843

In [41]:
# View a random sample
random.sample(pages_and_chunks, k=1)

[{'page_number': 84,
  'sentence_chunk': '“Blood Flow Through the Heart” by OpenStax College / CC BY 3.0 The cardiovascular system is one of the eleven organ systems of the human body. Its main function is to transport nutrients to cells and wastes from cells (Figure 2.12 “Cardiovascular Transportation of Nutrients”). This system consists of the heart, blood, and blood vessels. The heart pumps the blood, and the blood is the transportation fluid. The transportation route to all tissues, a highly intricate blood-vessel network, comprises arteries, veins, and capillaries. Nutrients absorbed in the small intestine travel mainly to the liver through the hepatic portal vein. From the liver, nutrients travel upward through the inferior vena cava blood vessel to the heart. The heart forcefully pumps the nutrient-rich blood first to the lungs to pick up some oxygen and then to all other cells in the body. Arteries become smaller and smaller on their way to cells, so that by the time blood reac

In [42]:
# Get stats about our chunks
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,1843.00,1843.00,1843.00,1843.00
mean,583.38,734.44,112.33,183.61
std,347.79,447.54,71.22,111.89
min,-41.00,12.00,3.00,3.00
25%,280.50,315.00,44.00,78.75
50%,586.00,746.00,114.00,186.50
75%,890.00,1118.50,173.00,279.62
max,1166.00,1831.00,297.00,457.75


In [43]:
# Show random chunks with under 30 tokens in length
min_token_length = 30
for row in df[df["chunk_token_count"] <= min_token_length].sample(5).iterrows():
    print(f'Chunk token count: {row[1]["chunk_token_count"]} | Text: {row[1]["sentence_chunk"]}')

Chunk token count: 24.5 | Text: view it online here: http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=130   Introduction | 149
Chunk token count: 24.25 | Text: There are several lecithin supplements on the market Nonessential and Essential Fatty Acids | 315
Chunk token count: 29.25 | Text: Abagovomab (monoclonal antibody) by Blake C / CC BY-SA 3.0 Figure 6.13 Antigens Protein’s Functions in the Body | 389
Chunk token count: 21.5 | Text: http://www.health.gov.fj/?page_id=1406. Accessed November 12, 2017. 652 | Introduction
Chunk token count: 12.75 | Text: PART VIII CHAPTER 8. ENERGY Chapter 8. Energy | 451


In [44]:
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
pages_and_chunks_over_min_token_len[:2]

[{'page_number': -39,
  'sentence_chunk': 'Human Nutrition: 2020 Edition UNIVERSITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN NUTRITION PROGRAM ALAN TITCHENAL, SKYLAR HARA, NOEMI ARCEO CAACBAY, WILLIAM MEINKE-LAU, YA-YUN YANG, MARIE KAINOA FIALKOWSKI REVILLA, JENNIFER DRAPER, GEMADY LANGFELDER, CHERYL GIBBY, CHYNA NICOLE CHUN, AND ALLISON CALABRESE',
  'chunk_char_count': 308,
  'chunk_word_count': 42,
  'chunk_token_count': 77.0},
 {'page_number': -38,
  'sentence_chunk': 'Human Nutrition: 2020 Edition by University of Hawai‘i at Mānoa Food Science and Human Nutrition Program is licensed under a Creative Commons Attribution 4.0 International License, except where otherwise noted.',
  'chunk_char_count': 210,
  'chunk_word_count': 30,
  'chunk_token_count': 52.5}]

In [45]:
# Requires !pip install sentence-transformers
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2",
                                      device="cpu") # choose the device to load the model to (note: GPU will often be *much* faster than CPU)

# Create a list of sentences to turn into numbers
sentences = [
    "The Sentences Transformers library provides an easy and open-source way to create embeddings.",
    "Sentences can be embedded one by one or as a list of strings.",
    "Embeddings are one of the most powerful concepts in machine learning!",
    "Learn to use embeddings well and you'll be well on your way to being an AI engineer."
]

# Sentences are encoded/embedded by calling model.encode()
embeddings = embedding_model.encode(sentences)
embeddings_dict = dict(zip(sentences, embeddings))

# See the embeddings
for sentence, embedding in embeddings_dict.items():
    print("Sentence:", sentence)
    print("Embedding:", embedding)
    print("")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Sentence: The Sentences Transformers library provides an easy and open-source way to create embeddings.
Embedding: [-2.07982846e-02  3.03164683e-02 -2.01217718e-02  6.86484724e-02
 -2.55256221e-02 -8.47688690e-03 -2.07212797e-04 -6.32377490e-02
  2.81607024e-02 -3.33353691e-02  3.02634146e-02  5.30721545e-02
 -5.03526740e-02  2.62288433e-02  3.33313979e-02 -4.51577418e-02
  3.63044925e-02 -1.37122103e-03 -1.20171420e-02  1.14947064e-02
  5.04510887e-02  4.70856726e-02  2.11913940e-02  5.14606275e-02
 -2.03746557e-02 -3.58889624e-02 -6.67784305e-04 -2.94393804e-02
  4.95859198e-02 -1.05639677e-02 -1.52014224e-02 -1.31761178e-03
  4.48197424e-02  1.56023623e-02  8.60379259e-07 -1.21394137e-03
 -2.37979181e-02 -9.09369497e-04  7.34484056e-03 -2.53930548e-03
  5.23370393e-02 -4.68043312e-02  1.66214425e-02  4.71579619e-02
 -4.15599309e-02  9.01964726e-04  3.60278003e-02  3.42213959e-02
  9.68227088e-02  5.94829172e-02 -1.64984446e-02 -3.51249129e-02
  5.92517434e-03 -7.07903819e-04 -2.4103

In [46]:
single_sentence = "Yo! How cool are embeddings?"
single_embedding = embedding_model.encode(single_sentence)
print(f"Sentence: {single_sentence}")
print(f"Embedding:\n{single_embedding}")
print(f"Embedding size: {single_embedding.shape}")

Sentence: Yo! How cool are embeddings?
Embedding:
[-1.97447781e-02 -4.51077055e-03 -4.98486497e-03  6.55444637e-02
 -9.87675786e-03  2.72835996e-02  3.66426297e-02 -3.30219255e-03
  8.50077160e-03  8.24952964e-03 -2.28497684e-02  4.02430147e-02
 -5.75200431e-02  6.33691847e-02  4.43207510e-02 -4.49506715e-02
  1.25284735e-02 -2.52011549e-02 -3.55292894e-02  1.29559003e-02
  8.67022946e-03 -1.92917921e-02  3.55637074e-03  1.89505592e-02
 -1.47128338e-02 -9.39847343e-03  7.64177507e-03  9.62185487e-03
 -5.98922372e-03 -3.90168726e-02 -5.47824800e-02 -5.67458663e-03
  1.11644613e-02  4.08067219e-02  1.76319099e-06  9.15306248e-03
 -8.77255946e-03  2.39382945e-02 -2.32784245e-02  8.04999620e-02
  3.19177061e-02  5.12603624e-03 -1.47708422e-02 -1.62524879e-02
 -6.03213012e-02 -4.35689948e-02  4.51211929e-02 -1.79053806e-02
  2.63366960e-02 -3.47866863e-02 -8.89172964e-03 -5.47675304e-02
 -1.24373008e-02 -2.38606539e-02  8.33496824e-02  5.71241751e-02
  1.13328183e-02 -1.49594918e-02  9.2037

In [47]:
%%time

# Uncomment to see how long it takes to create embeddings on CPU
# # Make sure the model is on the CPU
# embedding_model.to("cpu")

# # Embed each chunk one by one
# for item in tqdm(pages_and_chunks_over_min_token_len):
#     item["embedding"] = embedding_model.encode(item["sentence_chunk"])

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 8.11 µs


In [49]:
%%time

# Send the model to the GPU
embedding_model.to("cuda") # requires a GPU installed, for reference on my local machine, I'm using a NVIDIA RTX 4090

# Create embeddings one by one on the GPU
for item in tqdm(pages_and_chunks_over_min_token_len):
    item["embedding"] = embedding_model.encode(item["sentence_chunk"])

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# Turn text chunks into a single list
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]

In [ ]:
%%time

# Embed all texts in batches
text_chunk_embeddings = embedding_model.encode(text_chunks,
                                               batch_size=32, # you can use different batch sizes here for speed/performance, I found 32 works well for this use case
                                               convert_to_tensor=True) # optional to return embeddings as tensor instead of array

text_chunk_embeddings

In [ ]:
# Save embeddings to file
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
embeddings_df_save_path = "text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

In [ ]:
# Import saved file and view
text_chunks_and_embedding_df_load = pd.read_csv(embeddings_df_save_path)
text_chunks_and_embedding_df_load.head()

In [ ]:
import random

import torch
import numpy as np
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

# Import texts and embedding df
text_chunks_and_embedding_df = pd.read_csv("text_chunks_and_embeddings_df.csv")

# Convert embedding column back to np.array (it got converted to string when it got saved to CSV)
text_chunks_and_embedding_df["embedding"] = text_chunks_and_embedding_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

# Convert texts and embedding df to list of dicts
pages_and_chunks = text_chunks_and_embedding_df.to_dict(orient="records")

# Convert embeddings to torch tensor and send to device (note: NumPy arrays are float64, torch tensors are float32 by default)
embeddings = torch.tensor(np.array(text_chunks_and_embedding_df["embedding"].tolist()), dtype=torch.float32).to(device)
embeddings.shape

In [ ]:
text_chunks_and_embedding_df.head()

In [ ]:
embeddings[0]

In [ ]:
from sentence_transformers import util, SentenceTransformer

embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2",
                                      device=device) # choose the device to load the model to

In [ ]:
# 1. Define the query
# Note: This could be anything. But since we're working with a nutrition textbook, we'll stick with nutrition-based queries.
query = "macronutrients functions"
print(f"Query: {query}")

# 2. Embed the query to the same numerical space as the text examples
# Note: It's important to embed your query with the same model you embedded your examples with.
query_embedding = embedding_model.encode(query, convert_to_tensor=True)

# 3. Get similarity scores with the dot product (we'll time this for fun)
from time import perf_counter as timer

start_time = timer()
dot_scores = util.dot_score(a=query_embedding, b=embeddings)[0]
end_time = timer()

print(f"Time take to get scores on {len(embeddings)} embeddings: {end_time-start_time:.5f} seconds.")

# 4. Get the top-k results (we'll keep this to 5)
top_results_dot_product = torch.topk(dot_scores, k=5)
top_results_dot_product

In [ ]:
larger_embeddings = torch.randn(100*embeddings.shape[0], 768).to(device)
print(f"Embeddings shape: {larger_embeddings.shape}")

# Perform dot product across 168,000 embeddings
start_time = timer()
dot_scores = util.dot_score(a=query_embedding, b=larger_embeddings)[0]
end_time = timer()

print(f"Time take to get scores on {len(larger_embeddings)} embeddings: {end_time-start_time:.5f} seconds.")

In [ ]:
# Define helper function to print wrapped text
import textwrap

def print_wrapped(text, wrap_length=80):
    wrapped_text = textwrap.fill(text, wrap_length)
    print(wrapped_text)

In [ ]:
print(f"Query: '{query}'\n")
print("Results:")
# Loop through zipped together scores and indicies from torch.topk
for score, idx in zip(top_results_dot_product[0], top_results_dot_product[1]):
    print(f"Score: {score:.4f}")
    # Print relevant sentence chunk (since the scores are in descending order, the most relevant chunk will be first)
    print("Text:")
    print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
    # Print the page number too so we can reference the textbook further (and check the results)
    print(f"Page number: {pages_and_chunks[idx]['page_number']}")
    print("\n")

In [ ]:
def retrieve_relevant_resources(query, embeddings, model=embedding_model, n_resources_to_return=5):
    """
    Takes a query, embeds it, compares it with document embeddings,
    and returns the top-k most relevant chunks.
    """
    query_embedding = model.encode(query, convert_to_tensor=True)
    query_embedding = query_embedding.to(embeddings.device)

    dot_scores = util.dot_score(query_embedding, embeddings)[0]

    scores, indices = torch.topk(dot_scores, k=n_resources_to_return)

    return scores, indices


def print_top_results_and_scores(query, embeddings, pages_and_chunks, n_resources_to_return=5):
    """
    Prints top-k retrieved chunks with their similarity scores.
    """
    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=embeddings,
        n_resources_to_return=n_resources_to_return
    )

    print(f"Query: {query}\n")
    print("Top retrieved chunks:\n")

    for score, idx in zip(scores, indices):
        idx = idx.item()
        print(f"Score: {score:.4f}")
        print("Text:")
        print_wrapped(pages_and_chunks[idx]["sentence_chunk"])
        print(f"Page number: {pages_and_chunks[idx]['page_number']}")
        print("-" * 80)

In [ ]:
query = "symptoms of pellagra"

print_top_results_and_scores(
    query=query,
    embeddings=embeddings,
    pages_and_chunks=pages_and_chunks,
    n_resources_to_return=5
)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("LLM loaded successfully.")

In [ ]:
def generate_llm_answer(input_text, max_new_tokens=256):
    messages = [
        {"role": "user", "content": input_text}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_device = next(llm_model.parameters()).device

    inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Only decode newly generated tokens, not the original prompt
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer


test_question = "What are macronutrients?"
answer = generate_llm_answer(test_question)

print("Question:", test_question)
print("Answer:")
print(answer)

In [ ]:
def prompt_formatter(query, context_items):
    """
    Formats retrieved PDF chunks into a prompt for the LLM.
    """

    context = "\n\n".join(
        [
            f"Context {i+1} | Page {item['page_number']} | Score {item['score']:.4f}\n"
            f"{item['sentence_chunk']}"
            for i, item in enumerate(context_items)
        ]
    )

    prompt = f"""
You are a helpful assistant answering questions based on a PDF document.

Use only the context below to answer the question.
Do not use outside knowledge.
If the answer is not available in the context, say:
"The document does not provide enough information."

Context:
{context}

Question:
{query}

Answer:
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return formatted_prompt

In [ ]:
def ask(query, n_resources_to_return=2, max_new_tokens=256):
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks from PDF
    2. Add retrieved chunks to the prompt as context
    3. Generate an answer using the LLM
    """

    # 1. Retrieve relevant chunks
    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=embeddings,
        n_resources_to_return=n_resources_to_return
    )

    # 2. Prepare context items
    context_items = []

    for score, idx in zip(scores, indices):
        idx = idx.item()
        item = pages_and_chunks[idx].copy()
        item["score"] = score.item()
        context_items.append(item)

    # 3. Format prompt with retrieved context
    prompt = prompt_formatter(
        query=query,
        context_items=context_items
    )

    # 4. Tokenize prompt
    model_device = next(llm_model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

    # 5. Generate answer
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # 6. Decode only newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer, context_items

In [ ]:
query = "What are the symptoms of pellagra?"

answer, context_items = ask(
    query=query,
    n_resources_to_return=2,
    max_new_tokens=256
)

print("Query:")
print(query)

print("\nRAG Answer:")
print(answer)

print("\nRetrieved Contexts:")
for item in context_items:
    print(f"Score: {item['score']:.4f}")
    print(f"Page number: {item['page_number']}")
    print_wrapped(item["sentence_chunk"])
    print("-" * 80)

In [ ]:
query = "What are macronutrients, and what roles do they play in the human body?"

answer, context_items = ask(
    query=query,
    n_resources_to_return=2,
    max_new_tokens=256
)

print("Query:")
print(query)

print("\nRAG Answer:")
print(answer)

print("\nRetrieved Contexts:")
for item in context_items:
    print(f"Score: {item['score']:.4f}")
    print(f"Page number: {item['page_number']}")
    print_wrapped(item["sentence_chunk"])
    print("-" * 80)

In [ ]:
import pandas as pd

evaluation_queries = [
    "What are the symptoms of pellagra?",
    "What are macronutrients, and what roles do they play in the human body?",
    "How does water help the human body?",
    "What is the difference between macronutrients and micronutrients?",
    "What are water-soluble vitamins?"
]

rag_results = []

for query in evaluation_queries:
    answer, context_items = ask(
        query=query,
        n_resources_to_return=2,
        max_new_tokens=256
    )

    retrieved_pages = [item["page_number"] for item in context_items]
    retrieval_scores = [round(item["score"], 4) for item in context_items]

    rag_results.append({
        "Query": query,
        "RAG Answer": answer,
        "Retrieved Pages": retrieved_pages,
        "Retrieval Scores": retrieval_scores
    })

rag_results_df = pd.DataFrame(rag_results)
rag_results_df

In [ ]:
nutrition_dataset = {
    "name": "Human Nutrition PDF",
    "pdf_path": "human-nutrition-text.pdf",
    "chunks": pages_and_chunks,
    "embeddings": embeddings
}

print(nutrition_dataset["name"])
print("Number of chunks:", len(nutrition_dataset["chunks"]))
print("Embedding shape:", nutrition_dataset["embeddings"].shape)

In [ ]:
import os
import requests

poetry_pdf_path = "leaves_of_grass.pdf"
poetry_url = "https://www.waltwhitman.com/leaves-of-grass.pdf"

if not os.path.exists(poetry_pdf_path):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(poetry_url, headers=headers)

    if response.status_code == 200:
        with open(poetry_pdf_path, "wb") as f:
            f.write(response.content)
        print("Poetry PDF downloaded successfully.")
    else:
        print("Failed to download PDF.")
        print("Status code:", response.status_code)
else:
    print("Poetry PDF already exists.")

In [ ]:
import fitz

doc = fitz.open(poetry_pdf_path)

print("Number of pages:", len(doc))

sample_page = doc.load_page(5)
sample_text = sample_page.get_text("text")

doc.close()

print(sample_text[:1500])

In [ ]:
import fitz

doc = fitz.open(poetry_pdf_path)

for page_num in [5, 10, 15, 20, 25, 30, 35, 40, 50]:
    page = doc.load_page(page_num)
    text = page.get_text("text")
    print("=" * 100)
    print("PAGE:", page_num)
    print(text[:1000])

doc.close()

In [ ]:
import fitz

search_terms = [
    "ONE'S-SELF I SING",
    "I celebrate myself",
    "Song of Myself",
    "BOOK I. INSCRIPTIONS"
]

doc = fitz.open(poetry_pdf_path)

for page_num in range(len(doc)):
    text = doc.load_page(page_num).get_text("text")
    lower_text = text.lower()

    if any(term.lower() in lower_text for term in search_terms):
        print("=" * 100)
        print("Possible poetry start page:", page_num)
        print(text[:1200])
        break

doc.close()

In [ ]:
import fitz
import re
import pandas as pd
from tqdm.auto import tqdm

def clean_poetry_text(text: str) -> str:
    """
    Cleans poetry text while preserving line breaks.
    """
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def should_skip_poetry_line(line: str) -> bool:
    """
    Removes table-of-contents lines, page numbers, URLs, and decorative lines.
    """
    line = line.strip()

    if line == "":
        return True

    if re.fullmatch(r"\d+", line):
        return True

    if len(line) <= 2:
        return True

    if "Free eBooks" in line or "WaltWhitman.com" in line:
        return True

    # Table of contents style lines: TITLE .......... 221
    if re.search(r"\.{5,}\s*\d+$", line):
        return True

    return False


def open_and_read_poetry_pdf(pdf_path: str, start_page: int = 0, end_page: int | None = None) -> list[dict]:
    """
    Reads a poetry PDF while preserving line structure.
    """
    doc = fitz.open(pdf_path)

    if end_page is None:
        end_page = len(doc)

    pages_and_texts = []

    for page_number in tqdm(range(start_page, end_page)):
        page = doc.load_page(page_number)
        text = page.get_text("text")
        text = clean_poetry_text(text)

        pages_and_texts.append({
            "page_number": page_number,
            "page_char_count": len(text),
            "page_word_count": len(text.split()),
            "page_token_count": len(text) / 4,
            "text": text
        })

    doc.close()
    return pages_and_texts


def build_poetry_rag_dataset_from_pdf(
    pdf_path: str,
    dataset_name: str,
    embedding_model,
    device,
    start_page: int = 0,
    end_page: int | None = None,
    lines_per_chunk: int = 10,
    min_token_length: int = 8
):
    """
    Builds a RAG dataset from a poetry PDF using line-based chunking.
    """

    pages_and_texts = open_and_read_poetry_pdf(
        pdf_path=pdf_path,
        start_page=start_page,
        end_page=end_page
    )

    poetry_chunks = []

    for page in tqdm(pages_and_texts):
        raw_lines = page["text"].split("\n")

        lines = []

        for line in raw_lines:
            line = line.strip()

            if should_skip_poetry_line(line):
                continue

            lines.append(line)

        for i in range(0, len(lines), lines_per_chunk):
            chunk_lines = lines[i:i + lines_per_chunk]
            chunk_text = "\n".join(chunk_lines).strip()

            if len(chunk_text) == 0:
                continue

            chunk_dict = {
                "page_number": page["page_number"],
                "sentence_chunk": chunk_text,
                "chunk_char_count": len(chunk_text),
                "chunk_word_count": len(chunk_text.split()),
                "chunk_token_count": len(chunk_text) / 4,
                "document_type": "poetry"
            }

            poetry_chunks.append(chunk_dict)

    df_poetry_chunks = pd.DataFrame(poetry_chunks)

    poetry_chunks_over_min_token_len = df_poetry_chunks[
        df_poetry_chunks["chunk_token_count"] > min_token_length
    ].to_dict(orient="records")

    text_chunks = [
        item["sentence_chunk"]
        for item in poetry_chunks_over_min_token_len
    ]

    poetry_embeddings = embedding_model.encode(
        text_chunks,
        batch_size=32,
        convert_to_tensor=True
    ).to(device)

    poetry_dataset = {
        "name": dataset_name,
        "pdf_path": pdf_path,
        "chunks": poetry_chunks_over_min_token_len,
        "embeddings": poetry_embeddings,
        "document_type": "poetry"
    }

    print(f"Dataset name: {dataset_name}")
    print(f"Document type: poetry")
    print(f"Start page: {start_page}")
    print(f"Number of chunks: {len(poetry_dataset['chunks'])}")
    print(f"Embedding shape: {poetry_dataset['embeddings'].shape}")

    return poetry_dataset

In [ ]:
poetry_dataset = build_poetry_rag_dataset_from_pdf(
    pdf_path=poetry_pdf_path,
    dataset_name="Leaves of Grass Poetry PDF",
    embedding_model=embedding_model,
    device=device,
    start_page=11,
    lines_per_chunk=10,
    min_token_length=8
)

In [ ]:
len(poetry_dataset["chunks"])

In [ ]:
import random

for item in random.sample(poetry_dataset["chunks"], k=5):
    print("Page:", item["page_number"])
    print("Token count:", round(item["chunk_token_count"], 2))
    print(item["sentence_chunk"])
    print("-" * 100)

In [ ]:
bad_terms = ["Free eBooks", "WaltWhitman.com", "Contents"]

bad_chunks = [
    item for item in poetry_dataset["chunks"]
    if any(term.lower() in item["sentence_chunk"].lower() for term in bad_terms)
]

print("Number of suspicious chunks:", len(bad_chunks))

for item in bad_chunks[:5]:
    print("Page:", item["page_number"])
    print(item["sentence_chunk"])
    print("-" * 80)

In [ ]:
poetry_df = pd.DataFrame(poetry_dataset["chunks"])

poetry_df[[
    "chunk_char_count",
    "chunk_word_count",
    "chunk_token_count"
]].describe().round(2)

In [ ]:
poetry_query = "Which lines mention death or mortality?"

print_top_results_and_scores(
    query=poetry_query,
    embeddings=poetry_dataset["embeddings"],
    pages_and_chunks=poetry_dataset["chunks"],
    n_resources_to_return=3
)

In [ ]:
poetry_query = "Which lines express loneliness or sadness?"

print_top_results_and_scores(
    query=poetry_query,
    embeddings=poetry_dataset["embeddings"],
    pages_and_chunks=poetry_dataset["chunks"],
    n_resources_to_return=3
)

In [ ]:
poetry_query = "Which lines connect love, democracy, and religion?"

print_top_results_and_scores(
    query=poetry_query,
    embeddings=poetry_dataset["embeddings"],
    pages_and_chunks=poetry_dataset["chunks"],
    n_resources_to_return=3
)

In [ ]:
def prompt_formatter_poetry(query, context_items):
    """
    Formats retrieved poetry excerpts into a literary RAG prompt.
    """

    context = "\n\n".join(
        [
            f"Excerpt {i+1} | Page {item['page_number']} | Score {item['score']:.4f}\n"
            f"{item['sentence_chunk']}"
            for i, item in enumerate(context_items)
        ]
    )

    prompt = f"""
You are a careful literary assistant answering questions based on retrieved excerpts from a poetry PDF.

Use only the excerpts below.
If you interpret a theme, emotion, or symbol, clearly say that it is an interpretation based on the retrieved lines.
Do not claim the poet's exact intention unless it is directly stated.
Do not use outside knowledge about the poet or the book.
If the excerpts are not enough to answer, say:
"The document does not provide enough information."

Retrieved poetry excerpts:
{context}

Question:
{query}

Answer:
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return formatted_prompt

In [ ]:
def ask_poetry(query, dataset, n_resources_to_return=3, max_new_tokens=256):
    """
    Full RAG pipeline for poetry:
    1. Retrieve relevant poetry excerpts
    2. Add retrieved excerpts to a literary prompt
    3. Generate an answer using the LLM
    """

    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=dataset["embeddings"],
        n_resources_to_return=n_resources_to_return
    )

    context_items = []

    for score, idx in zip(scores, indices):
        idx = idx.item()
        item = dataset["chunks"][idx].copy()
        item["score"] = score.item()
        context_items.append(item)

    prompt = prompt_formatter_poetry(
        query=query,
        context_items=context_items
    )

    model_device = next(llm_model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer, context_items

In [ ]:
query = "Which lines mention death or mortality?"

answer, context_items = ask_poetry(
    query=query,
    dataset=poetry_dataset,
    n_resources_to_return=3,
    max_new_tokens=256
)

print("Query:")
print(query)

print("\nPoetry RAG Answer:")
print(answer)

print("\nRetrieved Poetry Excerpts:")
for item in context_items:
    print(f"Score: {item['score']:.4f}")
    print(f"Page number: {item['page_number']}")
    print(item["sentence_chunk"])
    print("-" * 100)

In [ ]:
query = "Which lines express loneliness or sadness?"

answer, context_items = ask_poetry(
    query=query,
    dataset=poetry_dataset,
    n_resources_to_return=3,
    max_new_tokens=256
)

print("Query:")
print(query)

print("\nPoetry RAG Answer:")
print(answer)

print("\nRetrieved Poetry Excerpts:")
for item in context_items:
    print(f"Score: {item['score']:.4f}")
    print(f"Page number: {item['page_number']}")
    print(item["sentence_chunk"])
    print("-" * 100)

In [ ]:
def ask_from_dataset(query, dataset, n_resources_to_return=2, max_new_tokens=256):
    """
    Generic factual RAG pipeline for a selected dataset.
    This will be used for the Human Nutrition PDF.
    """

    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=dataset["embeddings"],
        n_resources_to_return=n_resources_to_return
    )

    context_items = []

    for score, idx in zip(scores, indices):
        idx = idx.item()
        item = dataset["chunks"][idx].copy()
        item["score"] = score.item()
        context_items.append(item)

    prompt = prompt_formatter(
        query=query,
        context_items=context_items
    )

    model_device = next(llm_model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer, context_items

In [ ]:
import pandas as pd

comparison_rows = []

# Factual textbook queries
nutrition_tests = [
    "What are the symptoms of pellagra?",
    "What are macronutrients, and what roles do they play in the human body?",
    "What is the difference between macronutrients and micronutrients?"
]

for query in nutrition_tests:
    answer, context_items = ask_from_dataset(
        query=query,
        dataset=nutrition_dataset,
        n_resources_to_return=2,
        max_new_tokens=256
    )

    comparison_rows.append({
        "Dataset": nutrition_dataset["name"],
        "Document Type": "Factual textbook",
        "Chunking Strategy": "Sentence-based chunking",
        "Query Type": "Factual question",
        "Query": query,
        "Chunk Count": len(nutrition_dataset["chunks"]),
        "Embedding Shape": str(tuple(nutrition_dataset["embeddings"].shape)),
        "Top Pages": [item["page_number"] for item in context_items],
        "Top Scores": [round(item["score"], 4) for item in context_items],
        "Answer Style": "Direct factual answer",
        "RAG Answer": answer
    })


# Literary poetry queries
poetry_tests = [
    "Which lines mention death or mortality?",
    "Which lines express loneliness or sadness?",
    "Which lines connect love, democracy, and religion?"
]

for query in poetry_tests:
    answer, context_items = ask_poetry(
        query=query,
        dataset=poetry_dataset,
        n_resources_to_return=3,
        max_new_tokens=256
    )

    comparison_rows.append({
        "Dataset": poetry_dataset["name"],
        "Document Type": "Literary poetry",
        "Chunking Strategy": "Line-based chunking",
        "Query Type": "Thematic / interpretive question",
        "Query": query,
        "Chunk Count": len(poetry_dataset["chunks"]),
        "Embedding Shape": str(tuple(poetry_dataset["embeddings"].shape)),
        "Top Pages": [item["page_number"] for item in context_items],
        "Top Scores": [round(item["score"], 4) for item in context_items],
        "Answer Style": "Interpretive answer based on retrieved excerpts",
        "RAG Answer": answer
    })

comparison_df = pd.DataFrame(comparison_rows)

pd.set_option("display.max_colwidth", 180)
comparison_df

In [ ]:
comparison_df.to_csv("nutrition_vs_poetry_rag_comparison.csv", index=False)
print("Comparison table saved.")

In [ ]:
pd.set_option("display.max_colwidth", None)
comparison_df

In [ ]:
compact_comparison_df = comparison_df[
    [
        "Dataset",
        "Document Type",
        "Chunking Strategy",
        "Query Type",
        "Query",
        "Chunk Count",
        "Top Scores",
        "Answer Style"
    ]
]

compact_comparison_df

In [ ]:
import os
import requests

second_pdf_path = "attention_is_all_you_need.pdf"

url = "https://arxiv.org/pdf/1706.03762.pdf"

if not os.path.exists(second_pdf_path):
    response = requests.get(url)
    with open(second_pdf_path, "wb") as f:
        f.write(response.content)
    print("Second PDF downloaded successfully.")
else:
    print("Second PDF already exists.")

In [ ]:
import fitz
import re
import pandas as pd
from tqdm.auto import tqdm

def text_formatter(text: str) -> str:
    return text.replace("\n", " ").replace("\xa0", " ").strip()


def open_and_read_pdf_v2(pdf_path: str, page_offset: int = 0) -> list[dict]:
    doc = fitz.open(pdf_path)
    pages_and_texts = []

    for page_number, page in tqdm(enumerate(doc), total=len(doc)):
        text = page.get_text()
        text = text_formatter(text)

        pages_and_texts.append({
            "page_number": page_number + page_offset,
            "page_char_count": len(text),
            "page_word_count": len(text.split(" ")),
            "page_sentence_count_raw": len(text.split(". ")),
            "page_token_count": len(text) / 4,
            "text": text
        })

    doc.close()
    return pages_and_texts


def split_list(input_list: list, slice_size: int) -> list[list[str]]:
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]


def build_rag_dataset_from_pdf(
    pdf_path: str,
    dataset_name: str,
    embedding_model,
    device,
    page_offset: int = 0,
    num_sentence_chunk_size: int = 10,
    min_token_length: int = 30
):
    """
    Turns a PDF into chunks and embeddings for RAG.
    """

    # 1. Read PDF
    pages_and_texts = open_and_read_pdf_v2(
        pdf_path=pdf_path,
        page_offset=page_offset
    )

    # 2. Sentence splitting
    for item in tqdm(pages_and_texts):
        item["sentences"] = list(nlp(item["text"]).sents)
        item["sentences"] = [str(sentence) for sentence in item["sentences"]]
        item["page_sentence_count_spacy"] = len(item["sentences"])

    # 3. Chunking
    for item in tqdm(pages_and_texts):
        item["sentence_chunks"] = split_list(
            input_list=item["sentences"],
            slice_size=num_sentence_chunk_size
        )
        item["num_chunks"] = len(item["sentence_chunks"])

    # 4. Convert chunks to list of dictionaries
    pages_and_chunks_new = []

    for item in tqdm(pages_and_texts):
        for sentence_chunk in item["sentence_chunks"]:
            chunk_dict = {}

            joined_sentence_chunk = "".join(sentence_chunk).replace("  ", " ").strip()
            joined_sentence_chunk = re.sub(r"\.([A-Z])", r". \1", joined_sentence_chunk)

            chunk_dict["page_number"] = item["page_number"]
            chunk_dict["sentence_chunk"] = joined_sentence_chunk
            chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
            chunk_dict["chunk_word_count"] = len(joined_sentence_chunk.split(" "))
            chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4

            pages_and_chunks_new.append(chunk_dict)

    # 5. Filter short chunks
    df_chunks = pd.DataFrame(pages_and_chunks_new)

    pages_and_chunks_over_min_token_len = df_chunks[
        df_chunks["chunk_token_count"] > min_token_length
    ].to_dict(orient="records")

    # 6. Create embeddings
    text_chunks = [
        item["sentence_chunk"]
        for item in pages_and_chunks_over_min_token_len
    ]

    chunk_embeddings = embedding_model.encode(
        text_chunks,
        batch_size=32,
        convert_to_tensor=True
    ).to(device)

    dataset = {
        "name": dataset_name,
        "pdf_path": pdf_path,
        "chunks": pages_and_chunks_over_min_token_len,
        "embeddings": chunk_embeddings
    }

    print(f"Dataset name: {dataset_name}")
    print(f"Number of chunks: {len(dataset['chunks'])}")
    print(f"Embedding shape: {dataset['embeddings'].shape}")

    return dataset

In [ ]:
attention_dataset = build_rag_dataset_from_pdf(
    pdf_path=second_pdf_path,
    dataset_name="Attention Is All You Need PDF",
    embedding_model=embedding_model,
    device=device,
    page_offset=0,
    num_sentence_chunk_size=10,
    min_token_length=30
)

In [ ]:
def ask_from_dataset(query, dataset, n_resources_to_return=2, max_new_tokens=256):
    """
    Full RAG pipeline for a selected dataset.
    """

    scores, indices = retrieve_relevant_resources(
        query=query,
        embeddings=dataset["embeddings"],
        n_resources_to_return=n_resources_to_return
    )

    context_items = []

    for score, idx in zip(scores, indices):
        idx = idx.item()
        item = dataset["chunks"][idx].copy()
        item["score"] = score.item()
        context_items.append(item)

    prompt = prompt_formatter(
        query=query,
        context_items=context_items
    )

    model_device = next(llm_model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer, context_items

compare 2 datasets

In [ ]:
comparison_tests = [
    {
        "dataset": nutrition_dataset,
        "query": "What are the symptoms of pellagra?"
    },
    {
        "dataset": nutrition_dataset,
        "query": "What are macronutrients, and what roles do they play in the human body?"
    },
    {
        "dataset": attention_dataset,
        "query": "What is self-attention?"
    },
    {
        "dataset": attention_dataset,
        "query": "What is the Transformer architecture?"
    }
]

comparison_results = []

for test in comparison_tests:
    dataset = test["dataset"]
    query = test["query"]

    answer, context_items = ask_from_dataset(
        query=query,
        dataset=dataset,
        n_resources_to_return=2,
        max_new_tokens=256
    )

    comparison_results.append({
        "Dataset": dataset["name"],
        "Query": query,
        "Number of Chunks": len(dataset["chunks"]),
        "Embedding Shape": str(tuple(dataset["embeddings"].shape)),
        "Top Retrieved Pages": [item["page_number"] for item in context_items],
        "Top Scores": [round(item["score"], 4) for item in context_items],
        "RAG Answer": answer
    })

comparison_df = pd.DataFrame(comparison_results)
comparison_df

cross-dataset sanity check

In [ ]:
query = "What are the symptoms of pellagra?"

answer, context_items = ask_from_dataset(
    query=query,
    dataset=attention_dataset,
    n_resources_to_return=2,
    max_new_tokens=256
)

print("Dataset:", attention_dataset["name"])
print("Query:", query)
print("\nAnswer:")
print(answer)

print("\nRetrieved Contexts:")
for item in context_items:
    print(f"Score: {item['score']:.4f}")
    print(f"Page number: {item['page_number']}")
    print_wrapped(item["sentence_chunk"])
    print("-" * 80)